# **Validación y explotación del modelo Corepulse**

Este notebook documenta el proceso posterior a la preparación del modelo dimensional del proyecto **Corepulse Sales Analytics**.

La fase anterior dejó preparados los scripts de transformación, las tablas finales del modelo y el script SQL de carga en Snowflake. En esta fase se valida que el modelo se haya cargado correctamente en base de datos y se preparan las bases para su explotación en herramientas de Business Intelligence.


## **1. Objetivo de esta fase**.

El objetivo principal de esta fase es comprobar que el modelo estrella cargado en Snowflake es consistente y está listo para ser conectado a herramientas como Power BI o Looker Studio.

Las validaciones se centran en:

- Comprobar que las tablas se han creado y cargado correctamente.
- Validar que las dimensiones no tienen claves duplicadas.
- Comprobar que la tabla de hechos no contiene claves huérfanas.
- Validar que la granularidad de la fact se mantiene correctamente.
- Dejar documentado el estado del modelo antes de construir dashboards.


---

## **2. Modelo cargado en Snowflake**.

El modelo sigue una estructura de **modelo estrella**, con una tabla de hechos central y varias dimensiones descriptivas.

### Tabla de hechos

- `fact_ventas`

### Dimensiones

- `dim_producto`
- `dim_categoria`
- `dim_proveedor`
- `dim_calendario`

La granularidad esperada de la fact es:

```text
1 fila = 1 producto + 1 semana de negocio
```

Esto significa que la combinación `product_id + year_week_key` debe ser única dentro de `fact_ventas`.


----

## **3. Validación de los datos cargados.**

### **3.1. Validaciones: conteo de filas por tabla.**

La primera comprobación sirve para confirmar que todas las tablas se han cargado y que el volumen de registros es coherente con lo esperado.

Esta validación no comprueba todavía relaciones ni duplicados; simplemente responde a la pregunta:

> ¿Se han creado y cargado las tablas principales del modelo?


```yaml
-- Conteo de filas por tabla
SELECT 'dim_categoria' AS tabla, COUNT(*) AS filas FROM dim_categoria
UNION ALL
SELECT 'dim_proveedor', COUNT(*) FROM dim_proveedor
UNION ALL
SELECT 'dim_producto', COUNT(*) FROM dim_producto
UNION ALL
SELECT 'dim_calendario', COUNT(*) FROM dim_calendario
UNION ALL
SELECT 'fact_ventas', COUNT(*) FROM fact_ventas;

### Resultado obtenido

| Tabla | Filas |
|---|---:|
| dim_categoria | 19 |
| dim_proveedor | 20 |
| dim_producto | 60 |
| dim_calendario | 159 |
| fact_ventas | 9540 |

### Interpretación

El modelo se ha cargado correctamente en Snowflake. La fact contiene 9540 registros, coherentes con una estructura semanal por producto.

La lectura de la fact es coherente con el grano esperado:

```text
60 productos × 159 semanas = 9540 filas
```

Por tanto, a nivel de volumen, la carga parece correcta.


### **3.2. Validación 2: duplicados en dimensiones.**

Las dimensiones deben tener una fila única por cada clave primaria lógica.

Aunque en Snowflake se pueden declarar claves primarias y foráneas, en tablas estándar estas restricciones funcionan principalmente como metadatos. Por eso es importante comprobar manualmente que las claves realmente se comportan como únicas.

En el caso de `dim_calendario`, cada `year_week_key` debe aparecer una sola vez.


```yaml

-- Duplicados en dim_calendario
SELECT 
    year_week_key,
    COUNT(*) AS n_filas
FROM dim_calendario
GROUP BY year_week_key
HAVING COUNT(*) > 1;


### Lógica de la query

- `GROUP BY year_week_key` agrupa todas las filas que pertenecen a la misma semana.
- `COUNT(*)` cuenta cuántas filas hay para cada semana.
- `HAVING COUNT(*) > 1` muestra únicamente las semanas que aparecen más de una vez.

La diferencia clave es:

```text
WHERE  → filtra filas antes de agrupar
HAVING → filtra grupos después de agrupar
```

En este caso se usa `HAVING` porque queremos filtrar grupos agregados, no filas individuales.

### Resultado obtenido

La consulta devuelve **0 filas**.

### Interpretación

No hay semanas duplicadas en `dim_calendario`. La clave `year_week_key` funciona correctamente como identificador único de semana.


### **3.3. Validación 3: claves huérfanas entre fact y calendario.**

Esta validación comprueba si existen registros en `fact_ventas` cuya semana no exista en `dim_calendario`.

Una clave huérfana aparecería si la fact contiene un `year_week_key` que no tiene correspondencia en la dimensión calendario.


```yaml

-- Claves huérfanas entre fact_ventas y dim_calendario
SELECT COUNT(*) AS fact_sin_calendario
FROM fact_ventas f
LEFT JOIN dim_calendario c
    ON f.year_week_key = c.year_week_key
WHERE c.year_week_key IS NULL;

### Lógica de la query

Se utiliza un `LEFT JOIN` desde `fact_ventas` hacia `dim_calendario`.

Esto conserva todas las filas de la fact. Si una fila de la fact no encuentra coincidencia en calendario, las columnas de `dim_calendario` quedan como `NULL`.

Por eso se filtra con:

```sql
WHERE c.year_week_key IS NULL
```

Esa condición identifica las filas de la fact que no han encontrado correspondencia en la dimensión.

### Resultado obtenido

`fact_sin_calendario = 0`

### Interpretación

Todas las semanas presentes en `fact_ventas` existen también en `dim_calendario`. No hay claves huérfanas en la relación temporal.


### **3.4. Validación 4: granularidad de la fact.**

La granularidad esperada de `fact_ventas` es:

```text
1 fila = 1 producto + 1 semana de negocio
```

Por tanto, no debe existir más de una fila para la misma combinación de `product_id` y `year_week_key`.


```yaml

-- Validación de granularidad de la fact
SELECT 
    product_id,
    year_week_key,
    COUNT(*) AS n_filas
FROM fact_ventas
GROUP BY product_id, year_week_key
HAVING COUNT(*) > 1;

### Lógica de la query

- `GROUP BY product_id, year_week_key` agrupa por cada combinación producto-semana.
- `COUNT(*)` cuenta cuántas filas existen para cada combinación.
- `HAVING COUNT(*) > 1` filtra únicamente las combinaciones repetidas.

Si esta query devolviera resultados, significaría que la fact tiene duplicados a nivel de grano.

### Resultado obtenido

La consulta devuelve **0 filas**.

### Interpretación

La fact respeta correctamente la granularidad definida. No existen duplicados por producto y semana.


### **3.5. Resumen de validaciones.**

| Validación | Resultado | Interpretación |
|---|---:|---|
| Conteo de tablas | Correcto | Las tablas se han cargado correctamente. |
| Duplicados en calendario | 0 filas | `year_week_key` es único en `dim_calendario`. |
| Fact sin calendario | 0 | Todas las semanas de la fact existen en calendario. |
| Duplicados producto-semana | 0 filas | La fact respeta su granularidad. |

Conclusión: el modelo cargado en Snowflake está correctamente estructurado y puede utilizarse como base para la fase de visualización.


----

## 8. Próximos pasos

Una vez validado el modelo en Snowflake, los siguientes pasos del proyecto son:

1. Conectar Snowflake con Power BI.
2. Revisar relaciones del modelo en Power BI.
3. Crear medidas DAX principales.
4. Diseñar las páginas del dashboard.
5. Replicar o adaptar el análisis en Looker Studio.
6. Documentar decisiones de diseño y visualización.

Este notebook se actualizará progresivamente conforme avance la fase de explotación analítica.


----

## **4. Diseño de la capa analítica en Snowflake.**

Una vez cargado y validado el modelo estrella en Snowflake, el siguiente paso consiste en decidir qué cálculos deben construirse directamente en la base de datos y cuáles deben dejarse para la herramienta de visualización.

Esta decisión es importante porque no todos los cálculos tienen la misma naturaleza:

- Algunos cálculos son estables y reutilizables.
- Otros dependen del contexto de filtros del usuario.
- Algunos deben servir tanto para Power BI como para Looker Studio.
- Otros solo tienen sentido dentro de una visualización concreta.

Por este motivo, se plantea una separación entre:

```text
Snowflake       → capa analítica reutilizable
Power BI        → métricas dinámicas dependientes del contexto visual
Looker Studio   → visualización y campos calculados simples

### **4.1. Cálculos que se preparan en Snowflake**. 



En Snowflake se prepararán aquellas lógicas que interesa reutilizar en distintas herramientas de BI o que resultan más complejas de mantener directamente en la capa visual.

Ejemplos:
- vistas enriquecidas con joins entre fact y dimensiones;
- rankings anuales de productos;
- rankings anuales de proveedores;
- clasificaciones de productos;
- clasificaciones de proveedores;
- comparativas anuales;
- cálculos base de evolución YoY;
- rankings previstos para escenarios de forecast.



### **4.2. Cálculos que se dejan en Power BI**.


Power BI se utilizará para cálculos que deben responder dinámicamente a los filtros del usuario.

Ejemplos:
- ranking dinámico según categoría seleccionada;
- ranking dinámico según proveedor;
- top N dinámico;
- contribución de productos dentro del contexto filtrado;
- medidas afectadas por slicers;
- variaciones según selección temporal;
- KPIs interactivos.

Esto se debe a que DAX recalcula las medidas en función del contexto visual. Por tanto, un ranking creado en Power BI con RANKX puede cambiar cuando el usuario filtra por categoría, proveedor, campaña o periodo.

### **4.3. Diferencia entre ranking fijo y ranking dinámico.** 

Un ranking calculado en Snowflake es un ranking fijo o precalculado.

Por ejemplo, si se calcula el ranking anual de productos para 2025, Snowflake ordena los productos según las ventas de ese año y guarda esa posición en una vista.

Este ranking no cambia automáticamente si después, en Power BI o Looker Studio, se filtra por una categoría concreta. El ranking seguirá reflejando la posición calculada en el contexto original definido en SQL.

Por ejemplo, 

Ranking global 2025:

Producto A → ranking 1

Producto B → ranking 2

Producto C → ranking 3

Si después se filtra una categoría en Power BI, el producto C podría seguir mostrando ranking 3 aunque dentro de esa categoría sea el segundo producto visible.


### **4.4. Uso de PARTITION BY en rankings anuales.** 

Para calcular rankings separados por año en Snowflake se utiliza PARTITION BY.

La lógica es:

```yaml

RANK() OVER (
    PARTITION BY year_label
    ORDER BY sales_value DESC
) AS ranking_producto_anual

Esto significa que el ranking se reinicia para cada año y, por tanto, snowflake calcula de forma independiente y sin necesidad de crear tres consultas distintas:

Ranking de productos 2024

Ranking de productos 2025

Ranking de productos 2026

### **4.5. Tratamiento de escenarios de forecast**.

En el modelo existen ventas reales para 2024 y 2025, pero para 2026 existen previsiones en tres escenarios:

- forecast base,
- forecast optimista, 
- forecast pesimista.

Por tanto, si se desea analizar el ranking previsto de productos en 2026, este debe calcularse por escenario. 

La opción más limpia es transformar las métricas en formato largo, creando una columna escenario. Para conseguirlo, la query debe calcular el ranking con:

```yaml

PARTITION BY year_label, escenario

Esto permite obtener un ranking independiente para cada combinación de año y escenario.

### **4.6. Por qué no se crean tres vistas separadas para 2026.**

Aunque sería posible crear una vista para cada escenario, no se considera la opción más mantenible. 

Crear una única vista en formato largo es más escalable porque:

- evita duplicar lógica, 
- permite filtrar por escenario en la herramienta BI, 
- facilita la comparación entre escenarios, 
- permite reutilizar la misma estructura para productos y proveedores y
- simplifica el mantenimiento del modelo. 

Por tanto, se prefiere una vista única con una columna escenario. 

### **4.7. Decisión final.**

La decisión adoptada es crear en Snowflake una capa analítica con rankings fijos y comparativas base, y dejar los rankings dinámicos para Power BI.

En Snowflake se crearán vistas como:

vw_ventas_base
vw_ranking_productos_anual_escenario
vw_ranking_proveedores_anual_escenario
vw_comparativa_ranking_productos_anual
vw_comparativa_ranking_proveedores_anual

En Power BI se crearán medidas DAX para los cálculos que deban cambiar según los filtros del usuario.

Esta separación permite construir un modelo más profesional, reutilizable y claro. 

### **4.8. Explicación de las vistas.**

#### ➡️ **Vista 1 - vw_ventas_base.**

Es la vista base de consumo. Esto es, una vista que una la tabla de hechos con las dimensiones para no tener que repetir joins en cada análisis. 

```yaml

CREATE OR REPLACE VIEW vw_ventas_base AS
SELECT
    f.product_id,
    f.category_id,
    f.provider_id,
    f.year_week_key,

    p.nombre,
    p.marca,
    p.cluster_final,
    p.perfil_comportamiento,
    p.unit_sale_price_reference,

    c.categoria,
    c.familia_categoria,
    c.formato_categoria,

    pr.proveedor,
    pr.pais,
    pr.ccaa,
    pr.tipo_proveedor,
    pr.lead_time_dias_sim,
    pr.pedido_minimo_sim,

    cal.year_label,
    cal.week_number_business,
    cal.week_start_date,
    cal.week_end_date,
    cal.week_label,
    cal.month_start_name,
    cal.quarter_start_label,
    cal.is_partial_week,
    cal.is_campaign_week,
    cal.campaign_name_primary,
    cal.campaign_group_primary,

    f.sales_units,
    f.forecast_base_units,
    f.forecast_optimista_units,
    f.forecast_pesimista_units,
    f.sales_value_estimated,
    f.forecast_base_value_estimated,
    f.forecast_optimista_value_estimated,
    f.forecast_pesimista_value_estimated,
    f.sales_units_adjusted,
    f.forecast_base_units_adjusted,
    f.forecast_optimista_units_adjusted,
    f.forecast_pesimista_units_adjusted,
    f.sales_value_estimated_adjusted,
    f.forecast_base_value_estimated_adjusted,
    f.forecast_optimista_value_estimated_adjusted,
    f.forecast_pesimista_value_estimated_adjusted

FROM fact_ventas f
LEFT JOIN dim_producto p
    ON f.product_id = p.product_id
LEFT JOIN dim_categoria c
    ON f.category_id = c.category_id
LEFT JOIN dim_proveedor pr
    ON f.provider_id = pr.provider_id
LEFT JOIN dim_calendario cal
    ON f.year_week_key = cal.year_week_key;


🟪 **1. Qué problema resuelve.**

La vista `vw_ventas_base` se crea para centralizar en una única consulta la unión entre la tabla de hechos `fact_ventas` y sus dimensiones principales.

Aunque el modelo está correctamente estructurado en formato estrella, para determinados análisis y herramientas de visualización puede ser útil disponer de una vista enriquecida que ya incluya los atributos descriptivos de producto, categoría, proveedor y calendario.

Esta vista evita tener que repetir los mismos `JOIN` en cada análisis posterior y funciona como una capa de consumo analítico sobre el modelo dimensional.


🟪 **2. Grano del resultado.**

La vista conserva la granularidad de la tabla de hechos:

**1 fila = 1 producto + 1 semana de negocio**


🟪 **3. Columnas principales generadas.**

La vista incorpora varios bloques de informacion:

- Claves del modelo.
    - product_id
    - category_id
    - provider_id
    - year_week_key

- Atributos de producto.
    - nombre
    - marca
    - clúster
    - perfil de comportamiento
    - precio unitario de referencia

- Atributos de categoría.
    - categoría
    - familia de categoría
    - formato de la categoría

- Atributos de proveedor.
    - proveedor
    - país
    - comunidad autónoma
    - tipo de proveedor
    - lead time simulado
    - pedido mínimo simulado

- Atributos temporales. 
    - año
    - número de sema
    - fecha de inicio y fin de semana
    - campaña principal
    - flag de semana parcial
    - flag de semana de campaña

- Métricas. 
    - unidades vendidas reales
    - forecast base
    - forecast optimista
    - forecast pesimista
    - valores estimados
    - métricas ajustadas para semanas parciales

🟪 **4. Lógica utilizada.**

La query parte de fact_ventas, que es la tabla central del modelo, y utiliza LEFT JOIN para incorporar los atributos de cada dimensión.

Se usa LEFT JOIN porque se quiere conservar siempre la totalidad de registros de la fact. Si alguna dimensión no encontrara correspondencia, la fila de la fact seguiría apareciendo, permitiendo detectar posibles problemas de claves huérfanas.

La estructura principal es:

```yaml
fact_ventas
    LEFT JOIN dim_producto
    LEFT JOIN dim_categoria
    LEFT JOIN dim_proveedor
    LEFT JOIN dim_calendario




🟪 **5. Decisión técnica.**

Aunque el modelo estrella se mantiene como estructura principal, se crea esta vista como una capa analítica adicional. Esta decisión permite separar dos niveles:

- Tablas base del modelo estrella → estructura relacional limpia
- Vista de consumo → tabla enriquecida para análisis y visualización

La vista vw_ventas_base es equivalente, conceptualmente, a una capa de exploración o consumo sobre el modelo. En herramientas como Looker podría asimilarse a una vista/explore base; en Snowflake se implementa directamente como una VIEW.

🟪 **6. Limitaciones.**

La vista depende de que las dimensiones tengan claves únicas. Si alguna dimensión tuviera duplicados en sus claves, los JOIN podrían multiplicar filas y alterar las métricas. 

Por este motivo, antes de utilizar esta vista se han llevado a cabo las siguientes validaciones:

- duplicados en dimensiones, 
- claves huérfanas y 
- granularidad de la fact. 

Además, aunque esta vista es cómoda para herramientas como Looker Studio, en Power BI puede ser preferible mantener también el modelo estrella con relaciones entre tablas para aprovechar mejor el motor semántico y las medidas DAX.

🟪 **7. Uso posterior.**

La vista vw_ventas_base servirá como base para construir otras vistas analíticas en Snowflake, como:

- rankings de productos, 
- rankings de proveedores, 
- clasificaciones, 
- análisis YoY, 
- comparativas entre escenarios, 
- vistas de consumo para Power BI o Data Studio. 

También puede utilizarse directamente como fuente en herramientas de BI cuando se quiera trabajar con una tabla ya enriquecida y preparada para visualización.

#### ➡️ **Vista 1 - vw_ventas_base.**

🟪 **1. Qué problema resuelve.**

🟪 **2. Grano del resultado.**

🟪 **3. Columnas principales generadas.**

🟪 **4. Lógica utilizada.**

🟪 **5. Decisión técnica.**

🟪 **6. Limitaciones.**

🟪 **7. Uso posterior.**

#### ➡️ **Vista 2 - vw_ranking_productos_anual_escenario.**

Con esta vista, se resuelven varias cosas a la vez:

- ranking 2024 con ventas reales, 
- ranking 2025 con ventas reales, 
- ranking 2026 con forecast base, 
- ranking 2026 con forecast optimista, 
- ranking 2026 con forecast pesimista, 
- contribución individual, 
- contribución acumulada. 


```yaml

CREATE OR REPLACE VIEW vw_ranking_productos_anual_escenario AS

WITH metricas_producto_anual AS (

    -- Ventas reales 2024 y 2025
    SELECT
        year_label,
        'Real' AS escenario,
        product_id,
        nombre,
        marca,
        categoria,
        proveedor,
        SUM(sales_units) AS units_value,
        SUM(sales_value_estimated) AS sales_value
    FROM vw_ventas_base
    WHERE year_label IN (2024, 2025)
    GROUP BY
        year_label,
        product_id,
        nombre,
        marca,
        categoria,
        proveedor

    UNION ALL

    -- Forecast 2026: escenario base
    SELECT
        year_label,
        'Forecast base' AS escenario,
        product_id,
        nombre,
        marca,
        categoria,
        proveedor,
        SUM(forecast_base_units) AS units_value,
        SUM(forecast_base_value_estimated) AS sales_value
    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        product_id,
        nombre,
        marca,
        categoria,
        proveedor

    UNION ALL

    -- Forecast 2026: escenario optimista
    SELECT
        year_label,
        'Forecast optimista' AS escenario,
        product_id,
        nombre,
        marca,
        categoria,
        proveedor,
        SUM(forecast_optimista_units) AS units_value,
        SUM(forecast_optimista_value_estimated) AS sales_value
    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        product_id,
        nombre,
        marca,
        categoria,
        proveedor

    UNION ALL

    -- Forecast 2026: escenario pesimista
    SELECT
        year_label,
        'Forecast pesimista' AS escenario,
        product_id,
        nombre,
        marca,
        categoria,
        proveedor,
        SUM(forecast_pesimista_units) AS units_value,
        SUM(forecast_pesimista_value_estimated) AS sales_value
    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        product_id,
        nombre,
        marca,
        categoria,
        proveedor
),

ranking AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY year_label, escenario
            ORDER BY sales_value DESC
        ) AS ranking_producto,

        sales_value
        / NULLIF(
            SUM(sales_value) OVER (
                PARTITION BY year_label, escenario
            ),
            0
        ) AS pct_contribucion,

        SUM(sales_value) OVER (
            PARTITION BY year_label, escenario
            ORDER BY sales_value DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
        / NULLIF(
            SUM(sales_value) OVER (
                PARTITION BY year_label, escenario
            ),
            0
        ) AS pct_contribucion_acumulada

    FROM metricas_producto_anual
)

SELECT *
FROM ranking;

🟪 **1. Qué problema resuelve.**

La vista `vw_ranking_productos_anual_escenario` se crea para calcular el ranking anual de productos teniendo en cuenta tanto los años con ventas reales como el año forecast.

El proyecto contiene:

- ventas reales para 2024;
- ventas reales para 2025;
- forecast para 2026 en tres escenarios: base, optimista y pesimista.

Por tanto, no basta con calcular un ranking único para 2026. Es necesario calcular un ranking independiente para cada escenario de forecast, ya que la posición de los productos puede variar según el escenario analizado.

Esta vista permite responder preguntas como:

- qué productos lideran las ventas reales en 2024;
- qué productos lideran las ventas reales en 2025;
- qué productos lideran el forecast base de 2026;
- si el ranking cambia en el escenario optimista;
- si el ranking cambia en el escenario pesimista;
- qué productos concentran mayor contribución sobre el total anual.

🟪 **2. Grano del resultado.**

La vista tiene granularidad:

1 fila = 1 producto + 1 año + 1 escenario


🟪 **3. Columnas principales generadas.**

La vista incluye:

- Identificación temporal y de escenario:
    - year_label
    - escenario

- Identificación del producto:
    - product_id
    - nombre
    - marca
    - categoria
    - proveedor

- Métricas agregadas:
    - units_value
    - sales_value

- Métricas analíticas: 
    - ranking_producto
    - pct_contribucion
    - pct_contribucion_acumulada

🟪 **4. Lógica utilizada.**

La query parte de la vista vw_ventas_base, que ya contiene la fact enriquecida con atributos de producto, categoría, proveedor y calendario.

El primer bloque, metricas_producto_anual, transforma las métricas reales y de forecast en una estructura común.

Para 2024 y 2025 se utilizan las ventas reales:

- SUM(sales_units)
- SUM(sales_value_estimated)

Para 2026 se utilizan las métricas de forecast:

- SUM(forecast_base_units)
- SUM(forecast_base_value_estimated)

- SUM(forecast_optimista_units)
- SUM(forecast_optimista_value_estimated)

- SUM(forecast_pesimista_units)
- SUM(forecast_pesimista_value_estimated)

Para unificar todos los casos, se crea una columna llamada escenario.

Esto permite que ventas reales y forecast tengan una misma estructura analítica.


🟪 **5. Uso de UNION ALL.**

Se usa UNION ALL para apilcar los distintos bloques de datos:

Ventas reales 2024-2025
+
Forecast base 2026
+
Forecast optimista 2026
+
Forecast pesimista 2026

La ventaja de este enfoque es que evita crear tres vistas separadas para cada escenario de 2026. Esto hace que la solución sea más mantenible y más fácil de explotar desde Power BI o Looker Studio.

🟪 **6. Uso de PARTITION BY.**

El ranking se calcula con:

```yaml

RANK() OVER (
    PARTITION BY year_label, escenario
    ORDER BY sales_value DESC
) AS ranking_producto

La cláusula PARTITION BY year_label, escenario indica que el ranking debe reiniciarse para cada combinación de año y escenario.

Es decir, Snowflake calcula rankings independientes para:

```yaml
2024 - Real
2025 - Real
2026 - Forecast base
2026 - Forecast optimista
2026 - Forecast pesimista


Sin PARTITION BY, Snowflake calcularía un ranking global mezclando años y escenarios, lo cual no tendría sentido analítico.

🟪 **7. Contribución individual.**

La contribución individual indica qué porcentaje representa cada producto sobre le total de ventas o forecast de su año y escenario. 

Por ejemplo, si un producto representa el 8% del total del escenario base 2026, su pct_contribucion será 0,08. 

Se usa NULLIF(...,0) para evitar divisiones entre cero. 

Se calcula como sigue:

```yaml
sales_value
/ NULLIF(
    SUM(sales_value) OVER (
        PARTITION BY year_label, escenario
    ),
    0
) AS pct_contribucion


🟪 **8. Contribución acumulada.**

La contribución acumulada se calcula ordenando los productos de mayor a menor valor:

```yaml
SUM(sales_value) OVER (
    PARTITION BY year_label, escenario
    ORDER BY sales_value DESC
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW #suma desde la primera fila del ranking hasta la fila actual. 
)
/
NULLIF(
    SUM(sales_value) OVER (
        PARTITION BY year_label, escenario
    ),
    0
) AS pct_contribucion_acumulada

Esta métrica permite analizar cuánto peso acumulan los productos según su posición en el ranking.

Es útil para identificar:
- productos tops,
- productos core,
- productos secundarios, 
- long tail, 
- concentración de ventas. 

🟪 **9. Decisión técnica.**

La decisión adoptada es crear el ranking de productos en Snowflake como una vista fija y reutilizable.

Esta vista no pretende sustituir a los rankings dinámicos que puedan crearse posteriormente en Power BI. Su objetivo es generar una capa analítica estable para comparar años y escenarios de forma consistente.

La vista es especialmente útil para Looker Studio, donde las opciones de cálculo dinámico son más limitadas que en Power BI.

🟪 **10. Limitaciones.**

El ranking generado en esta vista es un ranking fijo o precalculado. Esto significa que el ranking no se recalcula automáticamente si en Power BI o Looker Studio se aplican filtros adicionales no contemplados en la partición.

Por ejemplo, si el ranking se calcula a nivel global por año y escenario, y después el usuario filtra una categoría concreta, el ranking seguirá mostrando la posición global del producto, no su posición dentro de esa categoría.

Para rankings completamente dinámicos se utilizarán medidas DAX en Power BI.


🟪 **11. Uso posterior.**

Esta vista se utilizará para:

- tablas de ranking anual de productos, 
- análisis top de productos por año, 
- comparación entre ventas reales y forecast, 
- análisis de escenarios de 2026, 
- cáclulo de clasificaciones de producto, 
- creación de vistas comparativas entre años. 

Además, servirá como base para construir vistas posteriores, como vw_comparativa_ranking_productos_anual.

#### ➡️ **Vista 3 - vw_ranking_proveedores_anual_escenario.**

Sirve para saber qué proveedores concentran más ventas reales o previstas según el año y el escenario. 


```yaml
CREATE OR REPLACE VIEW vw_ranking_proveedores_anual_escenario AS

WITH metricas_proveedor_anual AS (

    -- Ventas reales 2024 y 2025
    SELECT
        year_label,
        'Real' AS escenario,
        provider_id,
        proveedor,
        pais,
        ccaa,
        tipo_proveedor,
        lead_time_dias_sim,
        pedido_minimo_sim,
        COUNT(DISTINCT product_id) AS n_productos,
        SUM(sales_units) AS units_value,
        SUM(sales_value_estimated) AS sales_value
    FROM vw_ventas_base
    WHERE year_label IN (2024, 2025)
    GROUP BY
        year_label,
        provider_id,
        proveedor,
        pais,
        ccaa,
        tipo_proveedor,
        lead_time_dias_sim,
        pedido_minimo_sim

    UNION ALL

    -- Forecast 2026: escenario base
    SELECT
        year_label,
        'Forecast base' AS escenario,
        provider_id,
        proveedor,
        pais,
        ccaa,
        tipo_proveedor,
        lead_time_dias_sim,
        pedido_minimo_sim,
        COUNT(DISTINCT product_id) AS n_productos,
        SUM(forecast_base_units) AS units_value,
        SUM(forecast_base_value_estimated) AS sales_value
    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        provider_id,
        proveedor,
        pais,
        ccaa,
        tipo_proveedor,
        lead_time_dias_sim,
        pedido_minimo_sim

    UNION ALL

    -- Forecast 2026: escenario optimista
    SELECT
        year_label,
        'Forecast optimista' AS escenario,
        provider_id,
        proveedor,
        pais,
        ccaa,
        tipo_proveedor,
        lead_time_dias_sim,
        pedido_minimo_sim,
        COUNT(DISTINCT product_id) AS n_productos,
        SUM(forecast_optimista_units) AS units_value,
        SUM(forecast_optimista_value_estimated) AS sales_value
    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        provider_id,
        proveedor,
        pais,
        ccaa,
        tipo_proveedor,
        lead_time_dias_sim,
        pedido_minimo_sim

    UNION ALL

    -- Forecast 2026: escenario pesimista
    SELECT
        year_label,
        'Forecast pesimista' AS escenario,
        provider_id,
        proveedor,
        pais,
        ccaa,
        tipo_proveedor,
        lead_time_dias_sim,
        pedido_minimo_sim,
        COUNT(DISTINCT product_id) AS n_productos,
        SUM(forecast_pesimista_units) AS units_value,
        SUM(forecast_pesimista_value_estimated) AS sales_value
    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        provider_id,
        proveedor,
        pais,
        ccaa,
        tipo_proveedor,
        lead_time_dias_sim,
        pedido_minimo_sim
),

ranking AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY year_label, escenario
            ORDER BY sales_value DESC
        ) AS ranking_proveedor,

        sales_value
        / NULLIF(
            SUM(sales_value) OVER (
                PARTITION BY year_label, escenario
            ),
            0
        ) AS pct_contribucion,

        SUM(sales_value) OVER (
            PARTITION BY year_label, escenario
            ORDER BY sales_value DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
        / NULLIF(
            SUM(sales_value) OVER (
                PARTITION BY year_label, escenario
            ),
            0
        ) AS pct_contribucion_acumulada

    FROM metricas_proveedor_anual
)

SELECT *
FROM ranking;



🟪 **1. Qué problema resuelve.**

La vista `vw_ranking_proveedores_anual_escenario` calcula el ranking anual de proveedores combinando años con ventas reales y el año de forecast.

El objetivo es identificar qué proveedores concentran mayor volumen de ventas o previsión en cada periodo analizado.

Esta vista permite responder preguntas como:

- qué proveedores generaron más ventas en 2024;
- qué proveedores lideran en 2025;
- qué proveedores tienen mayor peso previsto en 2026;
- si el ranking de proveedores cambia según el escenario base, optimista o pesimista;
- qué proveedores concentran mayor porcentaje del valor total.

🟪 **2. Grano del resultado.**

La vista tiene granularidad:

1 fila = 1 proveedor + 1 año + 1 escenario

🟪 **3. Columnas principales generadas.**

La vista incluye:

- Identificación temporal y de escenario.
    - year_label
    - escenario

- Identificación del proveedor. 
    - provider_id
    - proveedor
    - pais
    - ccaa
    - tipo_proveedor

- Atributos operativos.
    - lead_time_dias_sim
    - pedido_minimo_sim
    - n_productos

- Métricas agregadas.
    - units_value
    - sales_value

- Métricas analíticas. 
    - ranking_proveedor
    - pct_contribucion
    - pct_contribucion_acumulada

🟪 **4. Lógica utilizada.**

La vista parte de vw_ventas_base, donde cada fila representa un producto por semana.

Para calcular el ranking de proveedores, se agregan las métricas a nivel proveedor, año y escenario.

Para 2024 y 2025 se utilizan las ventas reales:

- SUM(sales_units)
- SUM(sales_value_estimated)

Para 2026 se utilizan las métricas de forecast:

- SUM(forecast_base_units)
- SUM(forecast_base_value_estimated)

- SUM(forecast_optimista_units)
- SUM(forecast_optimista_value_estimated)

- SUM(forecast_pesimista_units)
- SUM(forecast_pesimista_value_estimated)

Al igual que en el ranking de productos, se crea la columna escenario para unificar ventas reales y forecast dentro de una misma estructura.

🟪 **5. Uso de UNION ALL.**

Se utiliza UNION ALL para apilar los distintos bloques:

Ventas reales 2024-2025
+
Forecast base 2026
+
Forecast optimista 2026
+
Forecast pesimista 2026

Este enfoque evita crear vistas separadas para cada escenario y permite explotar todos los escenarios desde una única fuente.

🟪 **6. Uso de PARTITION BY.**

El ranking se calcula con:

```yaml
RANK() OVER (
    PARTITION BY year_label, escenario
    ORDER BY sales_value DESC
) AS ranking_proveedor

Esto indica que el ranking se reinicia para cada combinación de año y escenario.

Por tanto, Snowflake calcula rankings independientes para:

- 2024 - Real
- 2025 - Real
- 2026 - Forecast base
- 2026 - Forecast optimista
- 2026 - Forecast pesimista


🟪 **7. Contribución individual.**

La contribución individual indica qué porcentaje representa cada proveedor sobre el total del año y escenario correspondiente.

Se calcula dividiendo el valor del proveedor entre el total de ventas o forecast de ese mismo año y escenario:


```yaml
sales_value
/
SUM(sales_value) OVER (
    PARTITION BY year_label, escenario
)

Se utiliza NULLIF(..., 0) para evitar divisiones entre cero.

🟪 **8. Contribución acumulada.**

La contribución acumulada permite saber cuánto peso acumulan los proveedores siguiendo el orden del ranking.

Esto es útil para detectar concentración de ventas en pocos proveedores.

Por ejemplo, 

Los 3 principales proveedores concentran el 65% del valor total.

Este tipo de análisis puede ayudar a detectar dependencia de proveedores concretos.

🟪 **9. Decisión técnica.**

La vista se crea en Snowflake porque el ranking anual de proveedores por escenario es una lógica reutilizable.

Además, Looker Studio tiene más limitaciones que Power BI para construir rankings complejos y comparativas entre escenarios. Por eso se prepara esta capa analítica previamente en Snowflake.

🟪 **10. Limitaciones.**

El ranking generado es fijo o precalculado.

Esto significa que no se recalcula dinámicamente si en Power BI o Looker Studio se aplican filtros adicionales no contemplados en la partición.

Por ejemplo, si el usuario filtra por una categoría concreta, el ranking seguirá mostrando la posición global del proveedor dentro del año y escenario, no necesariamente su posición dentro de esa categoría.

Para rankings completamente dinámicos se utilizarán medidas DAX en Power BI.

🟪 **11. Uso posterior.**

Esta vista se utilizará para:

- ranking anual de proveedores, 
- análisis de concentración por proveedor, 
- comparación de proveedores entre años, 
- comparación de proveedores según escenarios para 2026, 
- identificación de proveedores clave, 
- posibles visualizaciones de dependencia o riesgo operativo. 

Además, servirá como base para constuir vw_comparativa_ranking_proveedores_anual

##### Matiz importante

Esta vista no solo sirve para “top proveedores”. También te ayuda a justificar decisiones de negocio tipo:

```text
¿Dependemos demasiado de pocos proveedores?
¿El forecast 2026 aumenta la concentración?
¿Qué proveedores ganan peso en el escenario optimista?
¿Qué proveedores pierden relevancia en el escenario pesimista?

#### ➡️ **Vista 4 - vw_comparativa_ranking_productos_anual.**

Esta vista parte de la anteriormente creada: vw_ranking_productos_anual_escenario

Sirve para comparar:

- 2024 Real → 2025 Real
- 2025 Real → 2026 Forecast base
- 2025 Real → 2026 Forecast optimista
- 2025 Real → 2026 Forecast pesimista

```yaml
CREATE OR REPLACE VIEW vw_comparativa_ranking_productos_anual AS

WITH ranking_base AS (
    SELECT *
    FROM vw_ranking_productos_anual_escenario
),

comparativa AS (
    SELECT
        actual.product_id,
        actual.nombre,
        actual.marca,
        actual.categoria,
        actual.proveedor,

        anterior.year_label AS year_anterior,
        actual.year_label AS year_actual,

        anterior.escenario AS escenario_anterior,
        actual.escenario AS escenario_actual,

        anterior.units_value AS units_value_anterior,
        actual.units_value AS units_value_actual,

        anterior.sales_value AS sales_value_anterior,
        actual.sales_value AS sales_value_actual,

        anterior.ranking_producto AS ranking_anterior,
        actual.ranking_producto AS ranking_actual,

        anterior.pct_contribucion AS pct_contribucion_anterior,
        actual.pct_contribucion AS pct_contribucion_actual,

        anterior.pct_contribucion_acumulada AS pct_contribucion_acumulada_anterior,
        actual.pct_contribucion_acumulada AS pct_contribucion_acumulada_actual

    FROM ranking_base actual
    LEFT JOIN ranking_base anterior
        ON actual.product_id = anterior.product_id
        AND actual.year_label = anterior.year_label + 1
        AND anterior.escenario = 'Real'

    WHERE actual.year_label IN (2025, 2026)
)

SELECT
    *,

    units_value_actual - units_value_anterior AS var_units_abs,

    (units_value_actual - units_value_anterior)
        / NULLIF(units_value_anterior, 0) AS var_units_pct,

    sales_value_actual - sales_value_anterior AS var_value_abs,

    (sales_value_actual - sales_value_anterior)
        / NULLIF(sales_value_anterior, 0) AS var_value_pct,

    ranking_actual - ranking_anterior AS variacion_ranking,

    pct_contribucion_actual - pct_contribucion_anterior AS var_pct_contribucion,

    CASE
        WHEN ranking_anterior IS NULL 
             AND ranking_actual IS NOT NULL
            THEN 'Nuevo en ranking'

        WHEN ranking_actual < ranking_anterior
            THEN 'Sube posiciones'

        WHEN ranking_actual > ranking_anterior
            THEN 'Baja posiciones'

        WHEN ranking_actual = ranking_anterior
            THEN 'Mantiene posición'

        ELSE 'Sin clasificar'
    END AS estado_ranking,

    CAST(year_anterior AS VARCHAR)
        || ' '
        || escenario_anterior
        || ' vs '
        || CAST(year_actual AS VARCHAR)
        || ' '
        || escenario_actual AS periodo_comparativo

FROM comparativa;

🟪 **1. Qué problema resuelve.**

La vista `vw_comparativa_ranking_productos_anual` se crea para comparar la evolución del ranking de productos entre años consecutivos.

Parte de la vista `vw_ranking_productos_anual_escenario`, donde ya se ha calculado el ranking anual de productos para ventas reales y escenarios de forecast.

Esta vista permite responder preguntas como:

- qué productos suben posiciones de 2024 a 2025;
- qué productos bajan posiciones;
- qué productos mantienen su posición;
- cómo cambia el ranking previsto en 2026 según el escenario;
- qué productos ganan o pierden peso relativo;
- qué productos aumentan o reducen su contribución sobre el total.

🟪 **2. Grano del resultado.**

La vista tiene granularidad:

1 fila = 1 producto + 1 periodo comparativo + 1 escenario actual

🟪 **3. Columnas principales generadas.**

La vista incorpora:

- Identificación de producto.
    - product_id
    - nombre
    - marca
    - categoria
    - proveedor

- Periodos comparados.
    - year_anterior
    - year_actual
    - escenario_anterior
    - escenario_actual
    - periodo_comparativo

- Métricas comparadas. 
    - units_value_anterior
    - units_value_actual
    - sales_value_anterior
    - sales_value_actual

- Rankings comparados.
    - ranking_anterior
    - ranking_actual
    - variacion_ranking
    - estado_ranking

- Contribución. 
    - pct_contribucion_anterior
    - pct_contribucion_actual
    - var_pct_contribucion

🟪 **4. Lógica utilizada.**

La vista utiliza un LEFT JOIN de la tabla de ranking contra sí misma. Este patrón se conoce como self join.

La lógica consiste en unir cada producto del año actual con el mismo producto del año anterior.

La condición principal es:

```yaml
actual.product_id = anterior.product_id
AND actual.year_label = anterior.year_label + 1

Esto permite comparar:

- 2025 con 2024
- 2026 con 2024

Además, se fuerza que el año anterior sea siempre el escenario real:

```yaml
AND anterior.escenario = 'Real'

Esto es importante porque 2025 se compara contra 2024 real, y los escenarios de 2026 se comparan contra 2025 real.

🟪 **5. Comparativas generadas.**

La vista genera automáticamente varias comparativas:

- 2024 Real vs 2025 Real
- 2025 Real vs 2026 Forecast base
- 2025 Real vs 2026 Forecast optimista
- 2025 Real vs 2026 Forecast pesimista

Esto permite analizar tanto la evolución histórica como la evolución prevista.

🟪 **6. Variación de unidades y valor.**

La variación absoluta se calcula como: *sales_value_actual - sales_value_anterior*

La variación porcentual se calcula como:

```yaml

(sales_value_actual - sales_value_anterior)
/
NULLIF(sales_value_anterior, 0)

🟪 **7. Variación de ranking.**

La variación de ranking se calcula como: *ranking_actual - ranking_anterior*

Esta métrica requiere una interpretación especial:

- Valor negativo → mejora de posición
- Valor positivo → empeora posición
- Valor 0        → mantiene posición

Aunque matemáticamente un valor negativo pueda parecer una caída, en ranking significa mejora porque el producto se acerca al puesto 1.

🟪 **8. Variación de ranking.**

Para facilitar la interpretación, se crea un campo textual llamado estado_ranking.

Este campo clasifica cada producto como:

- Sube posiciones
- Baja posiciones
- Mantiene posición
- Nuevo en ranking
- Sin clasificar

Esto permite construir visualizaciones más intuitivas en Power BI o Looker Studio.

🟪 **9. Decisión técnica.**

La comparativa se construye en Snowflake porque es una lógica reutilizable y relativamente compleja para mantener directamente en Looker Studio.

Prepararla en Snowflake permite disponer de una vista ya lista para análisis comparativo, reduciendo la complejidad en la capa de visualización.

Esta vista actúa como una tabla analítica derivada, equivalente conceptualmente a una NDT o tabla derivada en Looker.

🟪 **10. Limitaciones.**

La comparación se basa en rankings precalculados en la vista anterior.

Por tanto, la variación de ranking compara posiciones dentro del contexto definido previamente:

año + escenario

Si después se aplican filtros adicionales en Power BI o Looker Studio, el ranking no se recalcula dinámicamente.

Para rankings totalmente dinámicos, se utilizarán medidas DAX en Power BI.

🟪 **11. Uso posterior.**

Esta vista se utilizará para:

- tablas comparativas de evolución de productos;
- análisis de subida o bajada de posiciones;
- identificación de productos con mayor crecimiento;
- identificación de productos con pérdida de relevancia;
- análisis de escenarios 2026;
- visualizaciones de cambio de ranking;
- filtros por periodo_comparativo.

En Looker Studio o Power BI, el campo periodo_comparativo permitirá seleccionar fácilmente la comparación deseada:

- 2024 Real vs 2025 Real
- 2025 Real vs 2026 Forecast base
- 2025 Real vs 2026 Forecast optimista
- 2025 Real vs 2026 Forecast pesimista

#### ➡️ **Vista 5 - vw_comparativa_ranking_proveedores_anual.**

Esta vista compara la evolución del ranking de proveedores entre años y escenarios.

Parte de: vw_ranking_proveedores_anual_escenario

Permite comparar:

- 2024 Real → 2025 Real
- 2025 Real → 2026 Forecast base
- 2025 Real → 2026 Forecast optimista
- 2025 Real → 2026 Forecast pesimista

```yaml
CREATE OR REPLACE VIEW vw_comparativa_ranking_proveedores_anual AS

WITH ranking_base AS (
    SELECT *
    FROM vw_ranking_proveedores_anual_escenario
),

comparativa AS (
    SELECT
        actual.provider_id,
        actual.proveedor,
        actual.pais,
        actual.ccaa,
        actual.tipo_proveedor,
        actual.lead_time_dias_sim,
        actual.pedido_minimo_sim,

        anterior.year_label AS year_anterior,
        actual.year_label AS year_actual,

        anterior.escenario AS escenario_anterior,
        actual.escenario AS escenario_actual,

        anterior.n_productos AS n_productos_anterior,
        actual.n_productos AS n_productos_actual,

        anterior.units_value AS units_value_anterior,
        actual.units_value AS units_value_actual,

        anterior.sales_value AS sales_value_anterior,
        actual.sales_value AS sales_value_actual,

        anterior.ranking_proveedor AS ranking_anterior,
        actual.ranking_proveedor AS ranking_actual,

        anterior.pct_contribucion AS pct_contribucion_anterior,
        actual.pct_contribucion AS pct_contribucion_actual,

        anterior.pct_contribucion_acumulada AS pct_contribucion_acumulada_anterior,
        actual.pct_contribucion_acumulada AS pct_contribucion_acumulada_actual

    FROM ranking_base actual
    LEFT JOIN ranking_base anterior
        ON actual.provider_id = anterior.provider_id
        AND actual.year_label = anterior.year_label + 1
        AND anterior.escenario = 'Real'

    WHERE actual.year_label IN (2025, 2026)
)

SELECT
    *,

    n_productos_actual - n_productos_anterior AS var_n_productos_abs,

    units_value_actual - units_value_anterior AS var_units_abs,

    (units_value_actual - units_value_anterior)
        / NULLIF(units_value_anterior, 0) AS var_units_pct,

    sales_value_actual - sales_value_anterior AS var_value_abs,

    (sales_value_actual - sales_value_anterior)
        / NULLIF(sales_value_anterior, 0) AS var_value_pct,

    ranking_actual - ranking_anterior AS variacion_ranking,

    pct_contribucion_actual - pct_contribucion_anterior AS var_pct_contribucion,

    CASE
        WHEN ranking_anterior IS NULL 
             AND ranking_actual IS NOT NULL
            THEN 'Nuevo en ranking'

        WHEN ranking_actual < ranking_anterior
            THEN 'Sube posiciones'

        WHEN ranking_actual > ranking_anterior
            THEN 'Baja posiciones'

        WHEN ranking_actual = ranking_anterior
            THEN 'Mantiene posición'

        ELSE 'Sin clasificar'
    END AS estado_ranking,

    CAST(year_anterior AS VARCHAR)
        || ' '
        || escenario_anterior
        || ' vs '
        || CAST(year_actual AS VARCHAR)
        || ' '
        || escenario_actual AS periodo_comparativo

FROM comparativa;

🟪 **1. Qué problema resuelve.**

La vista `vw_comparativa_ranking_proveedores_anual` se crea para comparar la evolución del ranking de proveedores entre años consecutivos y escenarios de forecast.

Parte de la vista `vw_ranking_proveedores_anual_escenario`, donde ya se ha calculado el ranking anual de proveedores para ventas reales y previsiones.

Esta vista permite responder preguntas como:

- qué proveedores suben posiciones de 2024 a 2025;
- qué proveedores pierden relevancia;
- qué proveedores mantienen una posición estable;
- qué proveedores ganan peso previsto en 2026;
- si el escenario optimista o pesimista cambia la importancia relativa de los proveedores;
- si aumenta o disminuye la dependencia de ciertos proveedores.

🟪 **2. Grano del resultado.**

La vista tiene granularidad:


1 fila = 1 proveedor + 1 periodo comparativo + 1 escenario actual

🟪 **3. Columnas principales generadas.**

La vista incorpora:

- Identificación del proveedor
    - provider_id
    - proveedor
    - pais
    - ccaa
    - tipo_proveedor
- Atributos operativos
    - lead_time_dias_sim
    - pedido_minimo_sim
- Periodos comparados
    - year_anterior
    - year_actual
    - escenario_anterior
    - escenario_actual
    - periodo_comparativo
- Métricas comparadas
    - n_productos_anterior
    - n_productos_actual
    - units_value_anterior
    - units_value_actual
    - sales_value_anterior
    - sales_value_actual
- Rankings comparados
    - ranking_anterior
    - ranking_actual
    - variacion_ranking
    - estado_ranking
- Contribución
    - pct_contribucion_anterior
    - pct_contribucion_actual
    - var_pct_contribucion

🟪 **4. Lógica utilizada.**

La vista utiliza un LEFT JOIN de la tabla de ranking de proveedores contra sí misma. Este patrón se conoce como self join.

La lógica consiste en unir cada proveedor del año actual con ese mismo proveedor en el año anterior.

La condición principal es:

actual.provider_id = anterior.provider_id
AND actual.year_label = anterior.year_label + 1

Esto permite comparar:

- 2025 con 2024
- 2026 con 2025

Además, se fuerza que el año anterior sea siempre el escenario real: AND anterior.escenario = 'Real'

Esto es importante porque los escenarios de 2026 se comparan contra 2025 real, no contra otro forecast.


🟪 **5. Comparativas generadas.**

La vista genera automáticamente estas comparativas:

- 2024 Real vs 2025 Real
- 2025 Real vs 2026 Forecast base
- 2025 Real vs 2026 Forecast optimista
- 2025 Real vs 2026 Forecast pesimista

Esto permite analizar tanto la evolución histórica como la evolución prevista por escenario.

🟪 **6. Variación de productos asociados.**

Además de comparar ventas y ranking, esta vista incluye: n_productos_actual - n_productos_anterior AS var_n_productos_abs

Esta métrica permite ver si un proveedor gana o pierde presencia dentro del catálogo analizado.

Por ejemplo, un proveedor puede subir en ventas no solo porque venda más por producto, sino porque tenga más productos asociados dentro del modelo.


🟪 **7. Variación de unidades y valor.**

La variación absoluta se calcula como: sales_value_actual - sales_value_anterior

La variación porcentual se calcula como:

```yaml
(sales_value_actual - sales_value_anterior)
/
NULLIF(sales_value_anterior, 0)

🟪 **8. Variación de ranking.**

La variación de ranking se calcula como: ranking_actual - ranking_anterior

Esta métrica requiere una interpretación específica:

- Valor negativo → mejora de posición
- Valor positivo → empeora posición
- Valor 0        → mantiene posición

🟪 **9. Estado del ranking.**

Para facilitar la interpretación, se crea un campo textual llamado estado_ranking.

Este campo clasifica cada proveedor como:

- Sube posiciones
- Baja posiciones
- Mantiene posición
- Nuevo en ranking
- Sin clasificar

Esto permite crear visualizaciones más claras y segmentar proveedores según su evolución.

🟪 **10. Decisión técnica.**

La comparativa de ranking de proveedores se construye en Snowflake porque es una lógica reutilizable, estable y más compleja de mantener directamente en Looker Studio.

Al dejarla preparada en Snowflake, la capa de visualización puede centrarse en consumir la información y representarla de forma clara.

Esta vista funciona como una tabla analítica derivada, equivalente conceptualmente a una NDT o tabla derivada en Looker.

🟪 **11. Limitaciones.**

La comparación se basa en rankings precalculados en la vista anterior.

Por tanto, la variación de ranking compara posiciones dentro del contexto definido previamente:

año + escenario

Si después se aplican filtros adicionales en Power BI o Looker Studio, el ranking no se recalcula dinámicamente.

Para rankings completamente dinámicos se utilizarán medidas DAX en Power BI.

🟪 **12. Uso posterior.**

Esta vista se utilizará para:

- análisis de evolución de proveedores;
- detección de proveedores que ganan peso;
- detección de proveedores que pierden relevancia;
- análisis de dependencia por proveedor;
- comparación de escenarios 2026;
- visualizaciones de cambio de ranking;
- filtros por periodo_comparativo.

En Power BI o Looker Studio, el campo periodo_comparativo permitirá seleccionar comparativas como:

- 2024 Real vs 2025 Real
- 2025 Real vs 2026 Forecast base
- 2025 Real vs 2026 Forecast optimista
- 2025 Real vs 2026 Forecast pesimista


#### Idea clave de esta vista

Esta vista no recalcula el ranking desde cero.

Hace esto:

1. Toma el ranking anual de proveedores ya calculado
2. Une cada proveedor con su año anterior
3. Calcula variaciones de ventas, contribución y posición
4. Clasifica si sube, baja o se mantiene

#### ➡️ **Vista 6 - vw_ventas_yoy_producto_semana.**

Esta vista compara cada producto contra el mismo producto en la misma semana del año anterior.

Genera comparativas como:

- 2024 Real → 2025 Real
- 2025 Real → 2026 Forecast base
- 2025 Real → 2026 Forecast optimista
- 2025 Real → 2026 Forecast pesimista

```yaml
CREATE OR REPLACE VIEW vw_ventas_yoy_producto_semana AS

WITH metricas_producto_semana AS (

    -- Ventas reales 2024 y 2025
    SELECT
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,

        'Real' AS escenario,

        product_id,
        nombre,
        marca,
        categoria,
        proveedor,

        sales_units AS units_value,
        sales_value_estimated AS sales_value,

        sales_units_adjusted AS units_value_adjusted,
        sales_value_estimated_adjusted AS sales_value_adjusted

    FROM vw_ventas_base
    WHERE year_label IN (2024, 2025)

    UNION ALL

    -- Forecast 2026: escenario base
    SELECT
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,

        'Forecast base' AS escenario,

        product_id,
        nombre,
        marca,
        categoria,
        proveedor,

        forecast_base_units AS units_value,
        forecast_base_value_estimated AS sales_value,

        forecast_base_units_adjusted AS units_value_adjusted,
        forecast_base_value_estimated_adjusted AS sales_value_adjusted

    FROM vw_ventas_base
    WHERE year_label = 2026

    UNION ALL

    -- Forecast 2026: escenario optimista
    SELECT
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,

        'Forecast optimista' AS escenario,

        product_id,
        nombre,
        marca,
        categoria,
        proveedor,

        forecast_optimista_units AS units_value,
        forecast_optimista_value_estimated AS sales_value,

        forecast_optimista_units_adjusted AS units_value_adjusted,
        forecast_optimista_value_estimated_adjusted AS sales_value_adjusted

    FROM vw_ventas_base
    WHERE year_label = 2026

    UNION ALL

    -- Forecast 2026: escenario pesimista
    SELECT
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,

        'Forecast pesimista' AS escenario,

        product_id,
        nombre,
        marca,
        categoria,
        proveedor,

        forecast_pesimista_units AS units_value,
        forecast_pesimista_value_estimated AS sales_value,

        forecast_pesimista_units_adjusted AS units_value_adjusted,
        forecast_pesimista_value_estimated_adjusted AS sales_value_adjusted

    FROM vw_ventas_base
    WHERE year_label = 2026
),

comparativa AS (
    SELECT
        actual.product_id,
        actual.nombre,
        actual.marca,
        actual.categoria,
        actual.proveedor,

        anterior.year_label AS year_anterior,
        actual.year_label AS year_actual,

        anterior.escenario AS escenario_anterior,
        actual.escenario AS escenario_actual,

        actual.week_number_business,
        anterior.year_week_key AS year_week_key_anterior,
        actual.year_week_key AS year_week_key_actual,

        anterior.week_label AS week_label_anterior,
        actual.week_label AS week_label_actual,

        actual.week_start_date,
        actual.week_end_date,

        actual.is_partial_week,
        actual.is_campaign_week,
        actual.campaign_name_primary,
        actual.campaign_group_primary,

        anterior.units_value AS units_value_anterior,
        actual.units_value AS units_value_actual,

        anterior.sales_value AS sales_value_anterior,
        actual.sales_value AS sales_value_actual,

        anterior.units_value_adjusted AS units_value_adjusted_anterior,
        actual.units_value_adjusted AS units_value_adjusted_actual,

        anterior.sales_value_adjusted AS sales_value_adjusted_anterior,
        actual.sales_value_adjusted AS sales_value_adjusted_actual

    FROM metricas_producto_semana actual
    LEFT JOIN metricas_producto_semana anterior
        ON actual.product_id = anterior.product_id
        AND actual.week_number_business = anterior.week_number_business
        AND actual.year_label = anterior.year_label + 1
        AND anterior.escenario = 'Real'

    WHERE actual.year_label IN (2025, 2026)
)

SELECT
    *,

    units_value_actual - units_value_anterior AS var_units_abs,

    (units_value_actual - units_value_anterior)
        / NULLIF(units_value_anterior, 0) AS var_units_pct,

    sales_value_actual - sales_value_anterior AS var_value_abs,

    (sales_value_actual - sales_value_anterior)
        / NULLIF(sales_value_anterior, 0) AS var_value_pct,

    units_value_adjusted_actual - units_value_adjusted_anterior 
        AS var_units_adjusted_abs,

    (units_value_adjusted_actual - units_value_adjusted_anterior)
        / NULLIF(units_value_adjusted_anterior, 0) AS var_units_adjusted_pct,

    sales_value_adjusted_actual - sales_value_adjusted_anterior 
        AS var_value_adjusted_abs,

    (sales_value_adjusted_actual - sales_value_adjusted_anterior)
        / NULLIF(sales_value_adjusted_anterior, 0) AS var_value_adjusted_pct,

    CAST(year_anterior AS VARCHAR)
        || ' '
        || escenario_anterior
        || ' vs '
        || CAST(year_actual AS VARCHAR)
        || ' '
        || escenario_actual AS periodo_comparativo

FROM comparativa;


🟪 **1. Qué problema resuelve.**

La vista `vw_ventas_yoy_producto_semana` se crea para analizar la evolución interanual de las ventas a nivel de producto y semana.

El objetivo es comparar cada producto con su comportamiento en la misma semana del año anterior.

Esta vista permite responder preguntas como:

- cómo evolucionan las ventas semanales de un producto de 2024 a 2025;
- qué productos crecen o caen semana a semana;
- cómo se comporta el forecast 2026 frente a las ventas reales de 2025;
- qué diferencias existen entre los escenarios base, optimista y pesimista;
- qué semanas presentan mayor variación interanual;
- cómo afectan campañas como Black Friday, Navidad o rebajas a la evolución semanal.

🟪 **2. Grano del resultado.**

La vista tiene granularidad:

1 fila = 1 producto + 1 semana + 1 periodo comparativo + 1 escenario actual

🟪 **3. Columnas principales generadas.**

- Identificación del producto
    - product_id
    - nombre
    - marca
    - categoria
    - proveedor
- Información temporal
    - year_anterior
    - year_actual
    - week_number_business
    - year_week_key_anterior
    - year_week_key_actual
    - week_label_anterior
    - week_label_actual
    - week_start_date
    - week_end_date
- Escenarios
    - escenario_anterior
    - escenario_actual
    - periodo_comparativo
- Contexto comercial
    - is_partial_week
    - is_campaign_week
    - campaign_name_primary
    - campaign_group_primary
- Métricas comparadas
    - units_value_anterior
    - units_value_actual
    - sales_value_anterior
    - sales_value_actual
- Métricas ajustadas
    - units_value_adjusted_anterior
    - units_value_adjusted_actual
    - sales_value_adjusted_anterior
    - sales_value_adjusted_actual
- Variaciones
    - var_units_abs
    - var_units_pct
    - var_value_abs
    - var_value_pct
    - var_units_adjusted_abs
    - var_units_adjusted_pct
    - var_value_adjusted_abs
    - var_value_adjusted_pct

🟪 **4. Lógica utilizada.**

La vista parte de vw_ventas_base.

El primer bloque, metricas_producto_semana, transforma las ventas reales y las previsiones de 2026 en una estructura común.

Para 2024 y 2025 se utilizan las ventas reales:

- sales_units
- sales_value_estimated
- sales_units_adjusted
- sales_value_estimated_adjusted

Para 2026 se utilizan las métricas de forecast en tres escenarios:

- forecast_base_units
- forecast_base_value_estimated

- forecast_optimista_units
- forecast_optimista_value_estimated

- forecast_pesimista_units
- forecast_pesimista_value_estimated

La columna escenario permite unificar ventas reales y previsiones dentro de una misma estructura.

🟪 **5. Uso de UNION ALL.**

Se utiliza UNION ALL para apilar cuatro bloques:

Ventas reales 2024-2025
+
Forecast base 2026
+
Forecast optimista 2026
+
Forecast pesimista 2026


Este enfoque permite comparar años reales y escenarios de forecast sin crear una vista distinta para cada escenario.

La estructura resultante permite trabajar siempre con las mismas columnas:

year_label
week_number_business
escenario
product_id
units_value
sales_value

🟪 **6. Uso del self join para comparar años.**

La comparación YoY se realiza mediante un LEFT JOIN de la tabla contra sí misma.

La parte principal es:

```yaml
actual.product_id = anterior.product_id
AND actual.week_number_business = anterior.week_number_business
AND actual.year_label = anterior.year_label + 1
AND anterior.escenario = 'Real'

Esto significa:

- se compara el mismo producto;
- se compara la misma semana de negocio;
- se compara el año actual contra el año anterior;
- el año anterior debe corresponder siempre a ventas reales.

Por tanto, la vista compara:

- 2025 Real contra 2024 Real
- 2026 Forecast base contra 2025 Real
- 2026 Forecast optimista contra 2025 Real
- 2026 Forecast pesimista contra 2025 Real


🟪 **7. Por qué se usa week_number_business.**

Se utiliza week_number_business para alinear semanas equivalentes entre años.

Por ejemplo:

Semana 48 de 2025 → Semana 48 de 2024
Semana 49 de 2026 → Semana 49 de 2025

Esto permite analizar evolución interanual a nivel semanal.

No obstante, hay que tener cuidado con las semanas parciales al inicio y al final del año, ya que pueden tener menos días y distorsionar la comparación visual.

Por este motivo, la vista conserva tanto las métricas originales como las métricas ajustadas.

🟪 **8. Métricas ajustadas y semanas parciales.**

Las semanas parciales pueden distorsionar la comparación porque no todas representan siete días completos.

Por ejemplo:

2025_W01 puede tener 5 días
2026_W01 puede tener 4 días

Si se comparan unidades reales sin ajustar, una semana con menos días podría parecer artificialmente peor.

Por eso la vista incluye métricas ajustadas:

units_value_adjusted
sales_value_adjusted

Estas métricas permiten realizar comparaciones más estables cuando existen semanas parciales.

En visualizaciones semanales, conviene indicar si se están utilizando valores reales o valores ajustados.

🟪 **9. Variaciones absolutas y porcentuales.**

La variación absoluta mide la diferencia directa entre el periodo actual y el anterior: sales_value_actual - sales_value_anterior

La variación porcentual mide esa diferencia en relación con el periodo anterior:

```yaml
(sales_value_actual - sales_value_anterior)
/
NULLIF(sales_value_anterior, 0)

🟪 **10. Decisión técnica.**

La vista se construye en Snowflake porque la lógica YoY semanal es reutilizable y puede ser compleja de recrear directamente en Looker Studio.

Además, al preparar esta vista en Snowflake, se facilita que Power BI y Looker Studio consuman una estructura ya preparada para análisis interanual.

Esta vista funciona como una tabla analítica derivada sobre el modelo estrella.

🟪 **11. Limitaciones.**

La comparación se basa en la alineación por número de semana de negocio.

Esto es útil para análisis YoY, pero no siempre garantiza una equivalencia exacta de calendario, especialmente cuando existen:

- semanas parciales;
- años con distinta distribución de días;
- campañas que pueden caer en semanas diferentes;
- festividades móviles, como Semana Santa.

Por tanto, las visualizaciones deben interpretarse teniendo en cuenta el contexto temporal y comercial.

🟪 **12. Uso posterior.**

Esta vista se utilizará para:

- gráficos de evolución semanal por producto;
- análisis YoY de ventas reales 2025 vs 2024;
- comparación de forecast 2026 contra ventas reales 2025;
- análisis de escenarios;
- detección de semanas con crecimiento o caída relevante;
- visualizaciones de campañas;
- análisis de productos concretos.

#### ➡️ **Vista 7 - vw_ventas_yoy_categoria_semana.**

Esta vista es muy parecida a la anterior, pero cambia el nivel de análisis. El producto + semana pasa a ser categoría + semana. 

Es decir, en vez de ver cómo evoluciona cada producto, analizamos cómo evoluciona cada categoría semana a semana respecto al año anterior.

```yaml
CREATE OR REPLACE VIEW vw_ventas_yoy_categoria_semana AS

WITH metricas_categoria_semana AS (

    -- Ventas reales 2024 y 2025
    SELECT
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,

        'Real' AS escenario,

        category_id,
        categoria,
        familia_categoria,
        formato_categoria,

        SUM(sales_units) AS units_value,
        SUM(sales_value_estimated) AS sales_value,

        SUM(sales_units_adjusted) AS units_value_adjusted,
        SUM(sales_value_estimated_adjusted) AS sales_value_adjusted

    FROM vw_ventas_base
    WHERE year_label IN (2024, 2025)
    GROUP BY
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,
        category_id,
        categoria,
        familia_categoria,
        formato_categoria

    UNION ALL

    -- Forecast 2026: escenario base
    SELECT
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,

        'Forecast base' AS escenario,

        category_id,
        categoria,
        familia_categoria,
        formato_categoria,

        SUM(forecast_base_units) AS units_value,
        SUM(forecast_base_value_estimated) AS sales_value,

        SUM(forecast_base_units_adjusted) AS units_value_adjusted,
        SUM(forecast_base_value_estimated_adjusted) AS sales_value_adjusted

    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,
        category_id,
        categoria,
        familia_categoria,
        formato_categoria

    UNION ALL

    -- Forecast 2026: escenario optimista
    SELECT
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,

        'Forecast optimista' AS escenario,

        category_id,
        categoria,
        familia_categoria,
        formato_categoria,

        SUM(forecast_optimista_units) AS units_value,
        SUM(forecast_optimista_value_estimated) AS sales_value,

        SUM(forecast_optimista_units_adjusted) AS units_value_adjusted,
        SUM(forecast_optimista_value_estimated_adjusted) AS sales_value_adjusted

    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,
        category_id,
        categoria,
        familia_categoria,
        formato_categoria

    UNION ALL

    -- Forecast 2026: escenario pesimista
    SELECT
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,

        'Forecast pesimista' AS escenario,

        category_id,
        categoria,
        familia_categoria,
        formato_categoria,

        SUM(forecast_pesimista_units) AS units_value,
        SUM(forecast_pesimista_value_estimated) AS sales_value,

        SUM(forecast_pesimista_units_adjusted) AS units_value_adjusted,
        SUM(forecast_pesimista_value_estimated_adjusted) AS sales_value_adjusted

    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,
        category_id,
        categoria,
        familia_categoria,
        formato_categoria
),

comparativa AS (
    SELECT
        actual.category_id,
        actual.categoria,
        actual.familia_categoria,
        actual.formato_categoria,

        anterior.year_label AS year_anterior,
        actual.year_label AS year_actual,

        anterior.escenario AS escenario_anterior,
        actual.escenario AS escenario_actual,

        actual.week_number_business,
        anterior.year_week_key AS year_week_key_anterior,
        actual.year_week_key AS year_week_key_actual,

        anterior.week_label AS week_label_anterior,
        actual.week_label AS week_label_actual,

        actual.week_start_date,
        actual.week_end_date,

        actual.is_partial_week,
        actual.is_campaign_week,
        actual.campaign_name_primary,
        actual.campaign_group_primary,

        anterior.units_value AS units_value_anterior,
        actual.units_value AS units_value_actual,

        anterior.sales_value AS sales_value_anterior,
        actual.sales_value AS sales_value_actual,

        anterior.units_value_adjusted AS units_value_adjusted_anterior,
        actual.units_value_adjusted AS units_value_adjusted_actual,

        anterior.sales_value_adjusted AS sales_value_adjusted_anterior,
        actual.sales_value_adjusted AS sales_value_adjusted_actual

    FROM metricas_categoria_semana actual
    LEFT JOIN metricas_categoria_semana anterior
        ON actual.category_id = anterior.category_id
        AND actual.week_number_business = anterior.week_number_business
        AND actual.year_label = anterior.year_label + 1
        AND anterior.escenario = 'Real'

    WHERE actual.year_label IN (2025, 2026)
)

SELECT
    *,

    units_value_actual - units_value_anterior AS var_units_abs,

    (units_value_actual - units_value_anterior)
        / NULLIF(units_value_anterior, 0) AS var_units_pct,

    sales_value_actual - sales_value_anterior AS var_value_abs,

    (sales_value_actual - sales_value_anterior)
        / NULLIF(sales_value_anterior, 0) AS var_value_pct,

    units_value_adjusted_actual - units_value_adjusted_anterior 
        AS var_units_adjusted_abs,

    (units_value_adjusted_actual - units_value_adjusted_anterior)
        / NULLIF(units_value_adjusted_anterior, 0) AS var_units_adjusted_pct,

    sales_value_adjusted_actual - sales_value_adjusted_anterior 
        AS var_value_adjusted_abs,

    (sales_value_adjusted_actual - sales_value_adjusted_anterior)
        / NULLIF(sales_value_adjusted_anterior, 0) AS var_value_adjusted_pct,

    CAST(year_anterior AS VARCHAR)
        || ' '
        || escenario_anterior
        || ' vs '
        || CAST(year_actual AS VARCHAR)
        || ' '
        || escenario_actual AS periodo_comparativo

FROM comparativa;

🟪 **1. Qué problema resuelve.**

La vista `vw_ventas_yoy_categoria_semana` se crea para analizar la evolución interanual de las ventas a nivel de categoría y semana.

Mientras que la vista anterior analizaba la evolución producto a producto, esta vista permite observar patrones agregados por categoría.

Esto ayuda a responder preguntas como:

- qué categorías crecen o caen respecto al año anterior;
- qué categorías presentan mejor evolución semanal;
- cómo se comportan las categorías principales en campañas;
- cómo evoluciona el forecast 2026 frente a las ventas reales de 2025;
- qué categorías son más sensibles a los distintos escenarios de previsión;
- si el crecimiento está concentrado en productos concretos o en una categoría completa.

🟪 **2. Grano del resultado.**

La vista tiene granularidad:

1 fila = 1 categoría + 1 semana + 1 periodo comparativo + 1 escenario actual

🟪 **3. Columnas principales generadas.**

La vista incorpora:

- Identificación de categoría
    - category_id
    - categoria
    - familia_categoria
    - formato_categoria
- Información temporal
    - year_anterior
    - year_actual
    - week_number_business
    - year_week_key_anterior
    - year_week_key_actual
    - week_label_anterior
    - week_label_actual
    - week_start_date
    - week_end_date
- Escenarios
    - escenario_anterior
    - escenario_actual
    - periodo_comparativo
- Contexto comercial
    - is_partial_week
    - is_campaign_week
    - campaign_name_primary
    - campaign_group_primary
- Métricas comparadas
    - units_value_anterior
    - units_value_actual
    - sales_value_anterior
    - sales_value_actual
- Métricas ajustadas
    - units_value_adjusted_anterior
    - units_value_adjusted_actual
    - sales_value_adjusted_anterior
    - sales_value_adjusted_actual
- Variaciones
    - var_units_abs
    - var_units_pct
    - var_value_abs
    - var_value_pct
    - var_units_adjusted_abs
    - var_units_adjusted_pct
    - var_value_adjusted_abs
    - var_value_adjusted_pct

🟪 **4. Lógica utilizada.**

La vista parte de vw_ventas_base.

El primer bloque, metricas_categoria_semana, agrega la información a nivel de categoría, año, semana y escenario.

Para 2024 y 2025 se utilizan las ventas reales:

- SUM(sales_units)
- SUM(sales_value_estimated)
- SUM(sales_units_adjusted)
- SUM(sales_value_estimated_adjusted)

Para 2026 se utilizan las previsiones en tres escenarios:

- SUM(forecast_base_units)
- SUM(forecast_base_value_estimated)

- SUM(forecast_optimista_units)
- SUM(forecast_optimista_value_estimated)

- SUM(forecast_pesimista_units)
- SUM(forecast_pesimista_value_estimated)

La columna escenario permite transformar las ventas reales y los forecasts en una estructura común.

🟪 **5. Decisión técnica.**

Se utiliza UNION ALL para apilar:

Ventas reales 2024-2025
+
Forecast base 2026
+
Forecast optimista 2026
+
Forecast pesimista 2026

Esto permite analizar todos los escenarios con la misma estructura y evita crear varias vistas separadas.

El resultado es una tabla lógica con columnas comunes:

- year_label
- week_number_business
- escenario
- category_id
- units_value
- sales_value

🟪 **6. Agregación por categoría.**

A diferencia de la vista vw_ventas_yoy_producto_semana, aquí las métricas se agregan con SUM().

Esto es necesario porque una categoría contiene varios productos.

Ejemplo:

Categoría Proteínas = suma de todos los productos de Proteínas

Por tanto, antes de comparar años, se calcula el total semanal de cada categoría.

🟪 **7. Uso del self join para comparar años.**

La comparación YoY se realiza mediante un LEFT JOIN de la tabla contra sí misma.

La condición principal es:

```yaml
actual.category_id = anterior.category_id
AND actual.week_number_business = anterior.week_number_business
AND actual.year_label = anterior.year_label + 1
AND anterior.escenario = 'Real'

Esto significa:

- se compara la misma categoría;
- se compara la misma semana de negocio;
- se compara el año actual contra el año anterior;
- el año anterior siempre corresponde a ventas reales.

Por tanto, la vista compara:

- 2025 Real contra 2024 Real
- 2026 Forecast base contra 2025 Real
- 2026 Forecast optimista contra 2025 Real
- 2026 Forecast pesimista contra 2025 Real

🟪 **8. Por qué se usa week_number_business.**

Se utiliza week_number_business para alinear semanas equivalentes entre años.

Por ejemplo:

Semana 48 de 2025 → Semana 48 de 2024
Semana 49 de 2026 → Semana 49 de 2025

Esto permite comparar la evolución semanal de cada categoría de forma homogénea.

Aun así, esta comparación debe interpretarse con cuidado si existen semanas parciales o campañas que cambian ligeramente de semana entre años.

🟪 **9. Métricas ajustadas y semanas parciales.**

Las semanas parciales pueden distorsionar el análisis porque no todas contienen siete días.

Por ejemplo:

2025_W01 puede tener 5 días
2026_W01 puede tener 4 días

Por ese motivo, la vista conserva métricas originales y ajustadas.

Las métricas ajustadas permiten comparar mejor semanas parciales, especialmente al inicio y final de año.

Esto es importante en visualizaciones semanales porque una caída en una semana parcial podría deberse simplemente a que esa semana contiene menos días, no a un peor comportamiento real de la categoría.

🟪 **10. Variaciones absolutas y porcentuales.**

La variación absoluta mide la diferencia directa entre el periodo actual y el anterior:

sales_value_actual - sales_value_anterior

La variación porcentual mide el cambio relativo frente al periodo anterior:

```yaml
(sales_value_actual - sales_value_anterior)
/
NULLIF(sales_value_anterior, 0)

🟪 **11. Decisión técnica.**

La vista se construye en Snowflake porque la lógica YoY semanal por categoría es reutilizable y puede ser compleja de mantener directamente en Looker Studio.

Además, trabajar con una vista agregada por categoría reduce la complejidad para la capa de visualización y permite crear gráficos comparativos más claros.

Esta vista funciona como una tabla analítica derivada sobre el modelo estrella.

🟪 **12. Limitaciones.**

La comparación se basa en el número de semana de negocio.

Esto permite una comparación ordenada y sencilla, pero puede tener limitaciones cuando:

- existen semanas parciales;
- una campaña cae en semanas distintas según el año;
- hay festividades móviles;
- se comparan semanas con distinto número de días;
- una categoría tiene pocos productos y puede ser sensible a valores extremos.

Por ello, las métricas ajustadas y los flags de campaña deben utilizarse como contexto interpretativo.

🟪 **13. Uso posterior.**

Esta vista se utilizará para:

- gráficos de evolución semanal por categoría;
- análisis YoY de categorías;
- comparación de ventas reales 2025 frente a 2024;
- comparación de forecast 2026 frente a ventas reales 2025;
- análisis de escenarios;
- detección de categorías con mayor crecimiento o caída;
- análisis de campañas por categoría;
- visualizaciones agregadas más estables que las de producto individual.

En Power BI o Looker Studio, el campo periodo_comparativo permitirá filtrar comparativas como:

- 2024 Real vs 2025 Real
- 2025 Real vs 2026 Forecast base
- 2025 Real vs 2026 Forecast optimista
- 2025 Real vs 2026 Forecast pesimista

> Las vistas analíticas se han creado correctamente en Snowflake y se han versionado en el repositorio mediante el archivo `sql/snowflake/10_create_analytics_views.sql`, para garantizar la reproducibilidad del modelo.

----